In [ ]:
# Importing all necessary libraries

import numpy as np
import pandas as pd

In [ ]:
### cell 0 ###

# Alternatively you can use this code to get the dataset
# url = "https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv"
birth_data = pd.read_csv("/home/dias-benchmarks/new_notebooks/us-birth/births.csv")
factor = 500
# replicate the entire DataFrame in blocks via one GPU gather
# build a single index array [0,1,…,n-1,0,1,…,n-1,…] and .take it
birth_data = birth_data.take(np.tile(np.arange(len(birth_data)), factor))
birth_data.sample(5, random_state=0)

In [ ]:
### cell 1 ###

birth_data.shape, birth_data.info()

In [ ]:
### cell 2 ###

birth_data = birth_data.fillna(0)
birth_data["day"] = birth_data["day"].astype(int)
birth_data.info()

In [ ]:
### cell 3 ###

birth_data.sample(5, random_state=0)

In [ ]:
### cell 4 ###

# Code to add the decade column
birth_data["decade"] = 10 * (birth_data["year"] // 10)
birth_data.head()

In [ ]:
### cell 5 ###

# take a look at male and female births as a function of decade
birth_data.groupby(["decade", "gender"])["births"].sum().unstack(fill_value=0)

In [ ]:
### cell 6 ###

# Code to present male and female mean(births) against decade numericaly
birth_data.groupby(["decade", "gender"])["births"].mean().unstack("gender")

In [ ]:
### cell 7 ###

years2check = [1988, 1989]
# Single GPU-accelerated filter instead of scanning twice
filtered = birth_data[birth_data["year"].isin(years2check)]
# Group on GPU and iterate only over the filtered years
for year, yearly_data in filtered.groupby("year"):
    print("Births data for -", year)
    print(yearly_data)

In [ ]:
### cell 8 ###

# Descriptive summary before removing outliers
birth_data.describe()

In [ ]:
### cell 9 ###

# Compute quartiles on GPU
quartiles = birth_data["births"].quantile([0.25, 0.5, 0.75])
q25, mu, q75 = quartiles.iloc[0], quartiles.iloc[1], quartiles.iloc[2]

# Sigma and bounds (still small Python scalars)
sig = 0.74 * (q75 - q25)
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig

# Filter on GPU via boolean indexing
births = birth_data[
    (birth_data["births"] > lower_bound) & (birth_data["births"] < upper_bound)
]

birth_data.shape, births.sample(5, random_state=0)

In [ ]:
### cell 10 ###

# Descriptive summary after removal of outliers
births.describe()

In [ ]:
### cell 11 ###

births.year.unique()

In [ ]:
### cell 12 ###

# create a datetime index from the year, month, day
births.index = pd.to_datetime(
    10000 * births.year + 100 * births.month + births.day, format="%Y%m%d"
)
births.head()

In [ ]:
### cell 13 ###

# Creating another column 'dayofweek' using dayofweek attribute of pandas DatetimeIndex
births["dayofweek"] = births.index.dayofweek
births.head()

In [ ]:
### cell 14 ###

mapping = {
    0: "Monday",
    1: "Tuesady",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}
births["dayofweek"] = births["dayofweek"].map(mapping)

In [ ]:
### cell 15 ###

births.head()

In [ ]:
### cell 16 ###

# Optimized for cudf: use GPU groupby instead of pivot_table
births_by_date = (
    births[["births"]].groupby([births.index.month, births.index.day]).mean()
)
births_by_date.sample(10, random_state=0)

In [ ]:
### cell 17 ###

births_by_date.shape

In [ ]:
### cell 18 ###

# Vectorized conversion to datetime index using GPU-accelerated to_datetime
births_by_date.index = pd.to_datetime(
    {
        "year": 2020,
        "month": births_by_date.index.get_level_values(0),
        "day": births_by_date.index.get_level_values(1),
    }
)
births_by_date.head()